In [ ]:
pip install faiss-cpu pandas open_clip_torch Pillow tqdm

  Using cached faiss_cpu-1.11.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (4.8 kB)
  Using cached open_clip_torch-2.32.0-py3-none-any.whl.metadata (31 kB)
  Using cached ftfy-6.3.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.5.147-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  U

In [ ]:
import os
import zipfile
import pickle
import pandas as pd
import numpy as np
import torch
import faiss
from PIL import Image
from tqdm import tqdm
import open_clip

In [ ]:
class CLIPIndexer:
    def __init__(self, image_folder_zip="images_test.zip", csv_path="result_test.csv", batch_size=64):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.image_folder_zip = image_folder_zip
        self.image_folder = self._unzip_correctly(image_folder_zip)
        self.csv_path = csv_path
        self.batch_size = batch_size

        # Charger le modèle CLIP
        self.model, _, self.preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
        self.model = self.model.to(self.device)
        self.model.eval()
        self.tokenizer = open_clip.get_tokenizer("ViT-B-32")

        self.image_embeddings = []
        self.text_embeddings = []
        self.faiss_image_mapping = []
        self.faiss_text_mapping = []

    def _unzip_correctly(self, zip_path):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            members = zip_ref.namelist()
            root_dirs = {name.split('/')[0] for name in members if '/' in name}
            extract_root = os.path.splitext(zip_path)[0]
            if len(root_dirs) == 1:
                zip_ref.extractall(".")
                return list(root_dirs)[0]
            else:
                zip_ref.extractall(extract_root)
                return extract_root

    def preprocess_image(self, image_path):
        image = Image.open(image_path).convert("RGB")
        return self.preprocess(image).unsqueeze(0)

    def index_data(self):
        df = pd.read_csv(self.csv_path, sep="|")
        df.columns = [col.strip() for col in df.columns]

        # --- Indexation des images par batch ---
        print("Indexation des images (batch)...")
        unique_images = df['image_name'].dropna().unique()
        img_batch, img_names = [], []

        for img_name in tqdm(unique_images):
            img_path = os.path.join(self.image_folder, img_name)
            if not os.path.exists(img_path):
                continue
            try:
                img = self.preprocess_image(img_path)
                img_batch.append(img)
                img_names.append(img_name)
            except Exception as e:
                print(f"Erreur image {img_name}: {e}")

            if len(img_batch) == self.batch_size:
                self._process_image_batch(img_batch, img_names)
                img_batch, img_names = [], []

        if img_batch:
            self._process_image_batch(img_batch, img_names)

        # --- Indexation des textes par batch ---
        print("Indexation des textes (batch)...")
        text_batch, text_img_names = [], []

        for _, row in tqdm(df.iterrows(), total=len(df)):
            img_name = row["image_name"]
            comment = str(row["comment"]).strip().lower()
            if comment:
                text_batch.append(comment)
                text_img_names.append(img_name)

            if len(text_batch) == self.batch_size:
                self._process_text_batch(text_batch, text_img_names)
                text_batch, text_img_names = [], []

        if text_batch:
            self._process_text_batch(text_batch, text_img_names)

        # --- FAISS indexes ---
        self._create_faiss_index()

        # Sauvegarde zip
        self.pack_indexes()

    def _process_image_batch(self, img_batch, img_names):
        batch_tensor = torch.cat(img_batch).to(self.device)
        with torch.no_grad():
            embeddings = self.model.encode_image(batch_tensor)
            embeddings /= embeddings.norm(dim=-1, keepdim=True)
        self.image_embeddings.extend(embeddings.cpu().numpy().astype("float32"))
        self.faiss_image_mapping.extend(img_names)

    def _process_text_batch(self, text_batch, img_names):
        tokenized = self.tokenizer(text_batch).to(self.device)
        with torch.no_grad():
            embeddings = self.model.encode_text(tokenized)
            embeddings /= embeddings.norm(dim=-1, keepdim=True)
        self.text_embeddings.extend(embeddings.cpu().numpy().astype("float32"))
        self.faiss_text_mapping.extend(img_names)

    def _create_faiss_index(self):
        if self.image_embeddings:
            arr = np.vstack(self.image_embeddings)
            self.index_image = faiss.IndexFlatL2(arr.shape[1])
            self.index_image.add(arr)
            faiss.write_index(self.index_image, "index_image.index")
            with open("faiss_image_mapping.pkl", "wb") as f:
                pickle.dump(self.faiss_image_mapping, f)
            print(f"📁 Index image créé avec {len(self.faiss_image_mapping)} images.")
        else:
            print("⚠️ Aucune image indexée.")

        if self.text_embeddings:
            arr = np.vstack(self.text_embeddings)
            self.index_text = faiss.IndexFlatL2(arr.shape[1])
            self.index_text.add(arr)
            faiss.write_index(self.index_text, "index_text.index")
            with open("faiss_text_mapping.pkl", "wb") as f:
                pickle.dump(self.faiss_text_mapping, f)
            print(f"📁 Index texte créé avec {len(self.faiss_text_mapping)} descriptions.")
        else:
            print("⚠️ Aucun texte indexé.")

    def pack_indexes(self, zip_filename="faiss_indexes.zip"):
        with zipfile.ZipFile(zip_filename, "w") as zipf:
            zipf.write("index_image.index")
            zipf.write("index_text.index")
            zipf.write("faiss_image_mapping.pkl")
            zipf.write("faiss_text_mapping.pkl")
        print(f"📦 Index compressé dans : {zip_filename}")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

shared_drive_path = "/content/drive/Shareddrives"
for root, dirs, files in os.walk(shared_drive_path):
    for d in dirs:
        print(os.path.join(root, d))

images =  "/content/drive/MyDrive/flikr/Images.zip"
result = "/content/drive/MyDrive/flikr/results.csv"

Mounted at /content/drive


In [ ]:
indexer = CLIPIndexer(image_folder_zip=images, csv_path=result)
indexer.index_data()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/open_clip/factory.py:388: UserWarning: These pretrained weights were trained with QuickGELU activation but the model config does not have that enabled. Consider using a model config with a "-quickgelu" suffix or enable with a flag.
  warnings.warn(


Indexation des images (batch)...


100%|██████████| 31783/31783 [06:13<00:00, 85.00it/s]


Indexation des textes (batch)...


100%|██████████| 158915/158915 [05:28<00:00, 483.29it/s]


📁 Index image créé avec 31783 images.
📁 Index texte créé avec 158915 descriptions.
📦 Index compressé dans : faiss_indexes.zip


In [ ]:
# Recherche textuelle
print(indexer.recherche_clip(query_text="girl"))

# Recherche par image
image_name = "1000092795.jpg"
image_path = os.path.join(indexer.image_folder, image_name)
print(indexer.recherche_clip(image_path=image_path))

NameError: name 'indexer' is not defined